<a href="https://colab.research.google.com/github/sborah53/Machine-Learning-for-Physical-Sciences/blob/main/MLPS_CNN_for_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Theoretical Foundations of Convolutional Neural Networks for Image Classification

This notebook implements a Convolutional Neural Network (CNN) to classify grayscale images of handwritten digits (MNIST). The goal of the model is to learn a mapping function $f_{\theta} : \mathscr{X} \rightarrow \mathscr{Y}$, where an input image $\mathbf{X} \in \mathbb{R}^{H \times W \times C}$ is mapped to a discrete class label $y \in \{0, 1, \dots, K-1\}$.

The architecture consists of feature extraction via convolutional and pooling operations, followed by non-linear activations, and culminates in fully connected layers for classification.

---

## 1. The Convolutional Layer

The core building block of a CNN is the convolutional layer, which utilizes learnable filters (or kernels) to extract spatial features such as edges, textures, and shapes.

Unlike dense layers that compute global combinations of inputs, convolutional layers perform a localized 2D cross-correlation operation. Given a 2D input feature map $\mathbf{I}$ and a 2D kernel $\mathbf{K}$ of size $m \times n$, the feature map $\mathbf{S}$ produced by the convolution operation is defined mathematically as:

$$S(i, j) = (\mathbf{I} * \mathbf{K})(i, j) = \sum_{m}\sum_{n} \mathbf{I}(i+m, j+n) \mathbf{K}(m, n)$$

**Key Properties:**
* **Sparse Interactions:** Each output unit only depends on a local receptive field of the input.
* **Parameter Sharing:** The same kernel weights $\mathbf{K}$ are swept across the entire input image, drastically reducing the number of parameters compared to a fully connected network.
* **Translation Equivariance:** If the input shifts, the resulting feature map shifts by the exact same amount.

---

## 2. Non-Linear Activation (ReLU)

Following the linear convolution operation, a non-linear activation function is applied element-wise. This model utilizes the Rectified Linear Unit (ReLU). Without non-linearities, any deep neural network would mathematically collapse into a single linear transformation.

The ReLU function is defined as:

$$f(x) = \max(0, x)$$

The gradient (derivative) of ReLU is computationally cheap and well-behaved, helping to mitigate the vanishing gradient problem during backpropagation:

$$f'(x) = \begin{cases} 1 & \text{if } x > 0 \\ 0 & \text{if } x \leq 0 \end{cases}$$

---

## 3. Spatial Downsampling (Max Pooling)

To reduce the spatial dimensions of the feature maps, control overfitting, and achieve approximate translation invariance, the network employs Max Pooling.

Given a pooling window of size $p \times p$ and a stride $s$, the max pooling operation maps a region of the input feature map $\mathbf{I}$ to a single maximum value in the output map $\mathbf{S}$:

$$S(i, j) = \max_{m, n \in [0, p-1]} \mathbf{I}(i \cdot s + m, j \cdot s + n)$$

This operation preserves the most prominent features (highest activations) while discarding less important spatial information.

---

## 4. Fully Connected (Dense) Layers

After extracting localized features through alternating convolution and pooling blocks, the multi-dimensional feature map is flattened into a 1D vector $\mathbf{x} \in \mathbb{R}^{D}$. This vector is passed through fully connected layers to aggregate the global context for classification.

The operation in a dense layer is an affine transformation followed by an activation:

$$\mathbf{z} = \mathbf{W}\mathbf{x} + \mathbf{b}$$

Where $\mathbf{W} \in \mathbb{R}^{M \times D}$ is the weight matrix, $\mathbf{b} \in \mathbb{R}^{M}$ is the bias vector, and $M$ is the number of output neurons.

---

## 5. Softmax and Cross-Entropy Loss

The final dense layer outputs a vector of raw, unnormalized scores called logits, $\mathbf{z} \in \mathbb{R}^{K}$, where $K=10$ (for digits 0-9).

### The Softmax Function
To interpret these logits as a valid probability distribution over the $K$ classes, the Softmax function is applied. For the $i$-th class, the predicted probability $\hat{y}_i$ is:

$$\hat{y}_i = \frac{\exp(z_i)}{\sum_{j=1}^{K} \exp(z_j)}$$

### Categorical Cross-Entropy
To optimize the network, we measure the divergence between the predicted probability distribution $\hat{\mathbf{y}}$ and the true target distribution $\mathbf{y}$ (which is one-hot encoded). The loss function utilized is Categorical Cross-Entropy:

$$\mathscr{L}(\theta) = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)$$

Because $y_i$ is a one-hot vector (it equals $1$ for the true class $c$, and $0$ otherwise), the loss simplifies to the negative log-likelihood of the true class:

$$\mathscr{L}(\theta) = -\log(\hat{y}_c)$$

---

## 6. Optimization: Stochastic Gradient Descent with Momentum

To minimize the loss function $\mathscr{L}(\theta)$ with respect to the network parameters $\theta$, the model uses Stochastic Gradient Descent (SGD) augmented with Momentum.

Standard SGD updates parameters in the opposite direction of the gradient. However, Momentum accelerates convergence by accumulating a velocity vector $\mathbf{v}$ in directions of persistent gradients, dampening oscillations in areas of high curvature.

The parameter update rules at time step $t$ are:

$$\mathbf{v}_{t} = \gamma \mathbf{v}_{t-1} + \eta \nabla_{\theta} \mathscr{L}(\theta_{t-1})$$
$$\theta_t = \theta_{t-1} - \mathbf{v}_t$$

Where:
* $\nabla_{\theta} \mathscr{L}(\theta_{t-1})$ is the gradient of the loss with respect to the parameters.
* $\eta$ is the learning rate.
* $\gamma \in [0, 1)$ is the momentum coefficient (typically $0.9$).

Through `jax.value_and_grad`, the gradients $\nabla_{\theta} \mathscr{L}$ are computed automatically via reverse-mode automatic differentiation (backpropagation), allowing the optimizer to iteratively refine the weights $\mathbf{W}$ and biases $\mathbf{b}$ across all layers.

In [ ]:
import jax
import jax.numpy as jnp
from flax import linen as nn
from flax.training import train_state
import optax
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

# Prevent TF from allocating GPU memory so JAX can have it all
tf.config.experimental.set_visible_devices([], "GPU")

# ------------------------------------------------------------------------------
# 1. Model Definition (Flax)
# ------------------------------------------------------------------------------
class CNN(nn.Module):
    """A simple CNN model loosely based on standard LeNet architecture."""

    @nn.compact
    def __call__(self, x):
        # x input shape: (batch_size, 28, 28, 1)

        # Convolutional Block 1
        x = nn.Conv(features=32, kernel_size=(3, 3))(x)
        x = nn.relu(x)
        x = nn.max_pool(x, window_shape=(2, 2), strides=(2, 2))

        # Convolutional Block 2
        x = nn.Conv(features=64, kernel_size=(3, 3))(x)
        x = nn.relu(x)
        x = nn.max_pool(x, window_shape=(2, 2), strides=(2, 2))

        # Flatten the spatial dimensions
        x = x.reshape((x.shape[0], -1))

        # Fully Connected Block
        x = nn.Dense(features=256)(x)
        x = nn.relu(x)

        # Output layer (10 classes for MNIST digits 0-9)
        x = nn.Dense(features=10)(x)
        return x

# ------------------------------------------------------------------------------
# 2. Loss and Metrics
# ------------------------------------------------------------------------------
def cross_entropy_loss(logits, labels):
    """Computes the cross entropy loss."""
    # Convert scalar labels to one-hot encoding
    labels_onehot = jax.nn.one_hot(labels, num_classes=10)
    # Optax provides a handy numerically stable cross entropy function
    loss = optax.softmax_cross_entropy(logits=logits, labels=labels_onehot)
    return jnp.mean(loss)

def compute_metrics(logits, labels):
    """Computes loss and accuracy metrics."""
    loss = cross_entropy_loss(logits, labels)
    accuracy = jnp.mean(jnp.argmax(logits, -1) == labels)
    return {'loss': loss, 'accuracy': accuracy}

# ------------------------------------------------------------------------------
# 3. Training and Evaluation Steps
# ------------------------------------------------------------------------------
def create_train_state(rng, learning_rate, momentum):
    """Creates the initial training state."""
    cnn = CNN()
    # Initialize the model weights by passing a dummy input
    params = cnn.init(rng, jnp.ones([1, 28, 28, 1]))['params']

    # Initialize the Optax optimizer (SGD with momentum)
    tx = optax.sgd(learning_rate, momentum)

    # Use AdamW for built-in weight decay (L2 Regularization)
    # tx = optax.adamw(learning_rate, weight_decay=1e-4)

    # TrainState is a handy Flax dataclass that holds the model params and optimizer state
    return train_state.TrainState.create(
        apply_fn=cnn.apply, params=params, tx=tx)

@jax.jit
def train_step(state, batch):
    """Trains for a single step (one batch). JIT compiled for speed."""

    def loss_fn(params):
        # Forward pass
        logits = state.apply_fn({'params': params}, batch['image'])
        # Compute loss
        loss = cross_entropy_loss(logits, batch['label'])
        return loss, logits

    # jax.value_and_grad returns both the loss value and the gradients
    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(state.params)

    # Update the parameters using the gradients
    state = state.apply_gradients(grads=grads)

    # Compute metrics for tracking
    metrics = compute_metrics(logits, batch['label'])
    return state, metrics

@jax.jit
def eval_step(state, batch):
    """Evaluates for a single step. JIT compiled."""
    logits = state.apply_fn({'params': state.params}, batch['image'])
    return compute_metrics(logits, batch['label'])

# ------------------------------------------------------------------------------
# 4. Data Loading (TensorFlow Datasets) & Visualization
# ------------------------------------------------------------------------------
def get_datasets(batch_size):
    """Load MNIST train, validation, and test datasets using highly optimized tf.data pipelines."""
    ds_builder = tfds.builder('mnist')
    ds_builder.download_and_prepare()

    def preprocess_fn(features):
        # Normalize images to [0, 1] and cast to float32
        image = tf.cast(features['image'], tf.float32) / 255.
        return {'image': image, 'label': features['label']}

    # Train dataset pipeline (80% of training data)
    train_ds = ds_builder.as_dataset(split='train[:80%]')
    train_ds = train_ds.map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.cache()
    train_ds = train_ds.shuffle(10000, reshuffle_each_iteration=True)
    train_ds = train_ds.batch(batch_size, drop_remainder=True)
    train_ds = train_ds.prefetch(tf.data.AUTOTUNE) # Asynchronously prefetch to GPU

    # Validation dataset pipeline (20% of training data)
    val_ds = ds_builder.as_dataset(split='train[80%:]')
    val_ds = val_ds.map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.batch(batch_size, drop_remainder=False)
    val_ds = val_ds.cache()
    val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

    # Test dataset pipeline
    test_ds = ds_builder.as_dataset(split='test')
    test_ds = test_ds.map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)
    test_ds = test_ds.batch(batch_size, drop_remainder=False)
    test_ds = test_ds.cache()
    test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds, test_ds

def plot_data_examples(ds_iter, num_examples=5):
    """Plots a few examples from the dataset."""
    batch = next(ds_iter)
    images, labels = batch['image'][:num_examples], batch['label'][:num_examples]

    fig, axes = plt.subplots(1, num_examples, figsize=(12, 3))
    for i in range(num_examples):
        axes[i].imshow(images[i].squeeze(), cmap='gray')
        axes[i].set_title(f"Label: {labels[i]}")
        axes[i].axis('off')
    plt.suptitle("Sample MNIST Data", fontsize=16)
    plt.show()

def plot_training_curves(history):
    """Plots the training and validation loss and accuracy curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history['train_loss']) + 1)

    ax1.plot(epochs, history['train_loss'], 'bo-', label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'ro-', label='Val Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()

    ax2.plot(epochs, history['train_acc'], 'bo-', label='Train Accuracy')
    ax2.plot(epochs, history['val_acc'], 'ro-', label='Val Accuracy')
    ax2.set_title('Training and Validation Accuracy')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.legend()

    plt.show()

def plot_predictions(state, ds_iter, num_examples=10):
    """Plots predictions vs reality for a few test samples."""
    batch = next(ds_iter)
    images, labels = batch['image'][:num_examples], batch['label'][:num_examples]

    # Get predictions
    logits = state.apply_fn({'params': state.params}, images)
    preds = np.argmax(logits, axis=-1)

    fig, axes = plt.subplots(2, num_examples // 2, figsize=(14, 8))
    axes = axes.flatten()
    for i in range(num_examples):
        axes[i].imshow(images[i].squeeze(), cmap='gray')
        color = 'green' if preds[i] == labels[i] else 'red'
        axes[i].set_title(f"Pred: {preds[i]} | True: {labels[i]}", color=color)
        axes[i].axis('off')
    plt.suptitle("Predictions vs Reality on Test Set", fontsize=16)
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------------------------
# 5. Main Training Loop
# ------------------------------------------------------------------------------
def main():
    # Hyperparameters
    learning_rate = 0.01
    momentum = 0.9
    num_epochs = 75
    batch_size = 128

    print("Loading datasets...")
    train_ds, val_ds, test_ds = get_datasets(batch_size)

    # Show some sample data before doing anything
    print("Showing some sample data...")
    train_iter = iter(tfds.as_numpy(train_ds))
    plot_data_examples(train_iter, num_examples=6)

    # Setup random key
    rng = jax.random.PRNGKey(0)
    rng, init_rng = jax.random.split(rng)

    print("Initializing model...")
    state = create_train_state(init_rng, learning_rate, momentum)

    # Dictionary to store history for plotting
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    print(f"Starting training for {num_epochs} epochs...")
    for epoch in range(1, num_epochs + 1):
        # Track metrics over the epoch
        batch_metrics = []

        # Re-initialize train iterator for each epoch
        train_iter = tfds.as_numpy(train_ds)
        for batch in train_iter:
            state, metrics = train_step(state, batch)
            batch_metrics.append(metrics)

        # Calculate epoch-level train metrics
        train_loss = np.mean([jax.device_get(m['loss']) for m in batch_metrics])
        train_acc = np.mean([jax.device_get(m['accuracy']) for m in batch_metrics])

        # Evaluate on the validation set
        val_metrics = []
        val_iter = tfds.as_numpy(val_ds)
        for batch in val_iter:
            metrics = eval_step(state, batch)
            val_metrics.append(metrics)

        val_loss = np.mean([jax.device_get(m['loss']) for m in val_metrics])
        val_acc = np.mean([jax.device_get(m['accuracy']) for m in val_metrics])

        # Record metrics in history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch:2d} | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")

    # Plot training & validation curves
    print("Plotting training curves...")
    plot_training_curves(history)

    # Evaluate on the completely unseen Test Set
    print("Evaluating on the held-out Test Set...")
    test_metrics = []
    test_iter = tfds.as_numpy(test_ds)
    for batch in test_iter:
        metrics = eval_step(state, batch)
        test_metrics.append(metrics)

    test_loss = np.mean([jax.device_get(m['loss']) for m in test_metrics])
    test_acc = np.mean([jax.device_get(m['accuracy']) for m in test_metrics])
    print(f"Final Test Result -> Loss: {test_loss:.4f}, Accuracy: {test_acc*100:.2f}%")

    # Plot final predictions
    print("Visualizing predictions vs reality...")
    test_iter = iter(tfds.as_numpy(test_ds))
    plot_predictions(state, test_iter, num_examples=10)

if __name__ == '__main__':
    main()